In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

In [ ]:
from urllib.parse import urlparse

import httpx
from langchain_core.tools import tool


def _docker_client() -> httpx.AsyncClient:
    """AsyncClient bound to the local Docker socket or DOCKER_HOST TCP host."""
    docker_host = os.environ.get("DOCKER_HOST") or "unix:///var/run/docker.sock"
    if docker_host.startswith("unix://"):
        transport = httpx.AsyncHTTPTransport(uds=docker_host[7:])
        base_url = "http://localhost"
    else:
        parsed = urlparse(docker_host)
        base_url = f"{parsed.scheme}://{parsed.hostname}:{parsed.port or 2375}"
        transport = None
    return httpx.AsyncClient(base_url=base_url, transport=transport, timeout=10)

import re

_SEPARATORS = re.compile(r"[-_\s.]+")

def _normalize(name: str) -> str:
    """Lowercase, strip separator chars: 'agent_api' == 'agent-api' == 'agentapi'."""
    return _SEPARATORS.sub("", str(name)).lower()

from datetime import UTC, datetime, timedelta


@tool
async def list_services() -> str:
    """List known infrastructure targets: real Docker container names and
    configured monitored endpoints. Call this FIRST when unsure of a name."""
    async with _docker_client() as client:
        response = await client.get("/v1.40/containers/json", params={"all": "true"})
        response.raise_for_status()
        containers = response.json()
    names = [(c.get("Names") or ["?"])[0].lstrip("/") for c in containers]
    endpoints = []
    for pair in os.environ.get("MONITORED_ENDPOINTS", "").split(","):
        if "=" in pair:
            n, u = (p.strip() for p in pair.split("=", 1))
            if n and u:
                endpoints.append(f"{n}={u}")
    lines = [
        "Containers: " + (", ".join(names) if names else "(none / docker unreachable)"),
        "Monitored endpoints: " + (", ".join(endpoints) if endpoints else "(none configured)"),
    ]
    return "\n".join(lines)


@tool
async def docker_status(container: str) -> str:
    """Return the state of a Docker container (read-only): state, status, image."""
    async with _docker_client() as client:
        response = await client.get("/v1.40/containers/json", params={"all": "true"})
        response.raise_for_status()
        containers = response.json()
    needle = _normalize(container)
    for c in containers:
        name = (c.get("Names") or ["?"])[0].lstrip("/")
        if needle in _normalize(name):
            return f"{name}: state={c.get('State')}, status={c.get('Status')}, image={c.get('Image')}"
    return f"(no container matching '{container}')"


@tool
async def query_loki(logql: str, limit: int = 100) -> str:
    """Run a LogQL query against Loki, scoped to the last 15 minutes."""
    max_limit = int(os.environ.get("LOKI_MAX_LIMIT", "500"))
    limit = max(1, min(limit, max_limit))
    end = datetime.now(UTC)
    start = end - timedelta(minutes=15)
    loki_url = os.environ.get("LOKI_URL", "http://localhost:3100")
    url = f"{loki_url}/loki/api/v1/query_range"
    headers = {}
    token = os.environ.get("LOKI_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
    params = {"query": logql, "start": start.isoformat(), "end": end.isoformat(), "limit": limit}
    async with httpx.AsyncClient(timeout=10.0, headers=headers) as client:
        response = await client.get(url, params=params)
        response.raise_for_status()
        return str(response.json())[:4000]


In [ ]:
from langchain_openai import ChatOpenAI

TOOLS = [list_services, docker_status, query_loki]

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ.get("LLM_API_KEY"),
    temperature=0,
).bind_tools(TOOLS)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

SYSTEM = "You are an infrastructure-operations investigator. Call list_services first, then check the service's container status and logs before concluding. If nothing in the evidence shows a problem, say so plainly instead of inventing a root cause."
question = "The api feels off. Can you check what's going on and find the root cause?"

TOOLS_BY_NAME = {t.name: t for t in TOOLS}

messages = [SystemMessage(SYSTEM), HumanMessage(question)]
for _ in range(8):
    reply = await llm.ainvoke(messages)
    messages.append(reply)
    if not reply.tool_calls:
        break
    for call in reply.tool_calls:
        try:
            result = await TOOLS_BY_NAME[call["name"]].ainvoke(call["args"] or {})
        except Exception as exc:  # noqa: BLE001 (tool errors are fed back to the model, never crash the loop)
            result = f"tool error: {exc}"
        messages.append(ToolMessage(tool_call_id=call["id"], content=str(result)[:4000]))

for m in messages:
    label = type(m).__name__
    body = m.content if not isinstance(m, ToolMessage) else f"[tool] {m.content}"
    print(f"{label}: {body}")